# NFL dedicated models (v1)

Each notebook in `ml/notebooks/v1/` is one model. They all share the same pipeline: look at FEATURES, train on 2023–24, test on 2025, register in `NFL_PROD_DB.ML`, write a pred table.

**Regression** predicts a number (points, yards, receptions). **Classification** (anytime TD only) predicts a probability.

Do not promote a version because it exists. Do not add FEATURES or ML to the Cortex agents. Do not loop all 14 cooks in one cell on the first pass.

| Model | Family | Task | Label | Notebook |
|---|---|---|---|---|
| `NFL_GAME_TOTAL` | game | regression | `label_total` | `nfl_game_total.ipynb` |
| `NFL_GAME_HOME_POINTS` | game | regression | `label_home_points` | `nfl_game_home_points.ipynb` |
| `NFL_GAME_AWAY_POINTS` | game | regression | `label_away_points` | `nfl_game_away_points.ipynb` |
| `NFL_GAME_MARGIN` | game | regression | `label_home_margin` | `nfl_game_margin.ipynb` |
| `NFL_GAME_HOME_NET_PASS` | game | regression | `label_home_net_pass` | `nfl_game_home_net_pass.ipynb` |
| `NFL_GAME_AWAY_NET_PASS` | game | regression | `label_away_net_pass` | `nfl_game_away_net_pass.ipynb` |
| `NFL_GAME_HOME_RUSH` | game | regression | `label_home_rush` | `nfl_game_home_rush.ipynb` |
| `NFL_GAME_AWAY_RUSH` | game | regression | `label_away_rush` | `nfl_game_away_rush.ipynb` |
| `NFL_PLAYER_PASSING_YARDS` | player | regression | `label_passing_yards` | `nfl_player_passing_yards.ipynb` |
| `NFL_PLAYER_RUSHING_YARDS` | player | regression | `label_rushing_yards` | `nfl_player_rushing_yards.ipynb` |
| `NFL_PLAYER_RECEIVING_YARDS` | player | regression | `label_receiving_yards` | `nfl_player_receiving_yards.ipynb` |
| `NFL_PLAYER_RECEPTIONS` | player | regression | `label_receptions` | `nfl_player_receptions.ipynb` |
| `NFL_PLAYER_PASSING_TDS` | player | regression | `label_passing_tds` | `nfl_player_passing_tds.ipynb` |
| `NFL_PLAYER_ANYTIME_TD` | player | classification | `label_anytime_td` | `nfl_player_anytime_td.ipynb` |

Deferred: first TD (needs play-level order), CLV vs close (2023–25 closes missing), `feat_player_prop_train` as a later FEATURES grain.

## Install the ML libraries

A model is just math that finds patterns in a table. We use two libraries:

- **scikit-learn** — trains the model on your laptop-shaped Python process (the notebook kernel).
- **snowflake-ml-python** — saves that trained model into Snowflake's Model Registry and can score it on a compute pool later.

This cell installs them with `uv pip` against Snowflake's own PyPI mirror (not public pypi.org, not Anaconda). The base image often already has them, but a weekend service restart wipes extra installs, so run this first every session. The `print` lines confirm versions so a later error is not a mystery missing-package.

In [ ]:
!uv pip install scikit-learn snowflake-ml-python

import importlib.metadata as md

print("sklearn", md.version("scikit-learn"))
print("snowflake-ml-python", md.version("snowflake-ml-python"))

## Connect the notebook to Snowflake and load this model's recipe

Two things happen here:

1. **Find `weekend_warriors_ml`.** The training code lives in the repo's `ml/` folder, not inside this notebook. We walk up from the current directory until we see `weekend_warriors_ml/pipeline.py`, then put that folder on `sys.path` so `import` works. If this raises, the Workspace is missing the `ml/` folder from the git pull.
2. **Open the kernel session.** `get_active_session()` is the Snowflake login this notebook already has. We do not type a password here.

`get_spec("NFL_GAME_TOTAL")` loads the recipe: which table, which columns are X (inputs), which column is the label (the answer we want to predict), and whether this is regression (a number) or classification (a yes/no probability). Nothing is trained yet.

In [ ]:
from pathlib import Path
import sys

here = Path.cwd().resolve()
for cand in (here, *here.parents):
    if (cand / "weekend_warriors_ml" / "pipeline.py").exists():
        if str(cand) not in sys.path:
            sys.path.insert(0, str(cand))
        break
else:
    raise FileNotFoundError(
        "weekend_warriors_ml not found. Put the repo ml/ folder in this Workspace."
    )

from snowflake.snowpark.context import get_active_session
from weekend_warriors_ml.specs import get_spec
from weekend_warriors_ml.pipeline import (
    cook,
    fit,
    inspect,
    log_experiment,
    register,
    score_batch,
    score_local,
)

SPEC = get_spec("NFL_GAME_TOTAL")
session = get_active_session()
print(SPEC.name, SPEC.task, len(SPEC.feature_columns), "features")
print(session.get_current_role(), session.get_current_warehouse())

## List every v1 recipe

`SPECS` is the registry in Python: name, family, task, label column, and which notebook owns it. This cell does not train anything. Skim it to see the fleet before you open a per-model notebook.

In [ ]:
from weekend_warriors_ml.specs import SPECS

for name, spec in SPECS.items():
    print(name, spec.family, spec.task, spec.label_column, spec.notebook)

## Cook one model end to end

`cook` runs inspect → fit → register → write the pred table. Change `MODEL` to any name from the table above. Start with `NFL_GAME_TOTAL` (already registered once) or a new game model.

This can take several minutes and **overwrites** that model's pred table. Do not put a `for` loop over all 14 names until you have watched one succeed. Skip `score_batch` unless you meant to start `ML_DEV_POOL`.

In [ ]:
MODEL = "NFL_GAME_TOTAL"
cook(session, MODEL)